# TD 6 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance.

> Les chiffres dépendent de la dernière mise à jour d'Eurostat : exécutez le notebook pour obtenir les valeurs du jour.

## Partie 1 — Charger et aligner

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"

URLS = {
    "vacances":  BASE + "jvs_q_nace2/Q.SA.B-S.TOTAL.JVR.FR?format=SDMX-CSV",
    "chomage":   BASE + "une_rt_q/Q.SA.Y15-74.PC_ACT.T.FR?format=SDMX-CSV",
    "inflation": BASE + "prc_hicp_manr/M.RCH_A.CP00.FR?format=SDMX-CSV",
}

def serie(url):
    d = pd.read_csv(url)[["TIME_PERIOD", "OBS_VALUE"]]
    return d.sort_values("TIME_PERIOD").reset_index(drop=True)

vac, cho, inf = serie(URLS["vacances"]), serie(URLS["chomage"]), serie(URLS["inflation"])

for nom, d in [("vacances", vac), ("chomage", cho), ("inflation", inf)]:
    print("%-10s %3d obs | %s -> %s" % (nom, len(d), d["TIME_PERIOD"].iloc[0], d["TIME_PERIOD"].iloc[-1]))

**Réponse 1.** Les emplois vacants commencent en **2011-Q2**, le chômage en 2003-Q1, l'inflation en 1997-01.

C'est donc la série des **emplois vacants qui limite** la courbe de Beveridge : quatorze ans, pas davantage. La courbe de Phillips, elle, peut remonter à 2003.

**Réponse 2.** Il faut **agréger l'inflation au trimestre**. On ne croise pas une série mensuelle avec une série trimestrielle : les dates ne correspondent pas, et `merge` ne trouverait aucune ligne commune.

**Réponse 3.** La **moyenne**.

L'inflation publiée ici est un *glissement annuel* : chaque valeur mensuelle dit déjà de combien les prix ont monté sur douze mois. En sommer douze donnerait un nombre sans signification — on additionnerait douze fois la même hausse.

> La règle : on somme les **flux** (ventes, heures travaillées), on moyenne les **taux et les niveaux**. Se tromper ici produit un résultat douze fois trop grand, sans aucun message d'erreur.

In [ ]:
inf["trimestre"] = (pd.PeriodIndex(inf["TIME_PERIOD"], freq="M")
                      .asfreq("Q").astype(str).str.replace("Q", "-Q"))

inf_t = inf.groupby("trimestre")["OBS_VALUE"].mean().reset_index()
inf_t.columns = ["TIME_PERIOD", "inflation"]
print(inf_t.head(3).to_string(index=False))

## Partie 2 — La courbe de Beveridge

In [ ]:
bev = vac.merge(cho, on="TIME_PERIOD", suffixes=("_vac", "_cho"))
bev.columns = ["trimestre", "vacances", "chomage"]
bev["annee"] = bev["trimestre"].str[:4].astype(int)

print(len(bev), "trimestres communs |", bev["trimestre"].iloc[0], "->", bev["trimestre"].iloc[-1])
print(bev[["vacances", "chomage"]].describe().round(2).to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# 1. le nuage brut
ax[0].scatter(bev["chomage"], bev["vacances"], s=18)
ax[0].set_title("Le nuage")

# 2. relie dans l'ordre du temps
ax[1].plot(bev["chomage"], bev["vacances"], marker="o", markersize=3, lw=1)
ax[1].set_title("Relié chronologiquement")

# 3. colore par periode
def periode(a):
    if a < 2020:  return "avant 2020"
    if a <= 2021: return "2020-2021"
    return "après 2021"

bev["periode"] = bev["annee"].apply(periode)
for nom in ["avant 2020", "2020-2021", "après 2021"]:
    g = bev[bev["periode"] == nom]
    ax[2].scatter(g["chomage"], g["vacances"], s=25, label=nom)
ax[2].set_title("Par période"); ax[2].legend()

for a in ax:
    a.set_xlabel("taux de chômage (%)"); a.set_ylabel("taux d'emplois vacants (%)")
plt.tight_layout(); plt.show()

print(bev.groupby("periode")[["chomage", "vacances"]].mean().round(2).to_string())

**Réponse 4.** Le nuage descend de la gauche vers la droite : **quand le chômage est élevé, les emplois vacants sont rares**, et inversement. C'est la relation décroissante attendue — la courbe de Beveridge.

Mais le nuage seul suggère une relation unique et stable, ce qu'il n'est pas.

**Réponse 5.** Relier les points dans l'ordre du temps fait apparaître que **la courbe n'est pas parcourue dans les deux sens le long d'une même ligne** : elle se déplace. Les points récents ne reviennent pas sur la trajectoire des années 2010.

> Un nuage de points fait perdre l'ordre chronologique. Sur une série temporelle, c'est presque toujours une information décisive.

**Réponse 6.** Le déplacement se produit **à partir de 2021** : le taux d'emplois vacants atteint des niveaux inédits alors que le chômage redescend seulement à ses niveaux d'avant-crise. Comparez les moyennes par période imprimées ci-dessus.

**Réponse 7.** À chômage donné, davantage d'emplois vacants signifie que **l'appariement fonctionne moins bien** : les entreprises cherchent, les chômeurs cherchent, et la rencontre se fait moins facilement qu'auparavant.

Les explications possibles sont nombreuses — inadéquation des qualifications, mobilité géographique, modification des exigences de part et d'autre, réallocation sectorielle après la crise sanitaire. **Le graphique établit le fait, il ne le explique pas.**

## Partie 3 — La courbe de Phillips

In [ ]:
phi = inf_t.merge(cho, on="TIME_PERIOD")
phi.columns = ["trimestre", "inflation", "chomage"]
phi["annee"] = phi["trimestre"].str[:4].astype(int)

COUPURE = 2021

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(phi["chomage"], phi["inflation"], s=18)
ax[0].set_title("Toute la période")

for avant, g in phi.groupby(phi["annee"] < COUPURE):
    ax[1].scatter(g["chomage"], g["inflation"], s=22,
                  label=f"avant {COUPURE}" if avant else f"{COUPURE} et après")
ax[1].set_title("Deux sous-périodes"); ax[1].legend()

for a in ax:
    a.set_xlabel("taux de chômage (%)"); a.set_ylabel("inflation (%)")
plt.tight_layout(); plt.show()

print("correlation, ensemble :", round(phi["chomage"].corr(phi["inflation"]), 3))
for avant, g in phi.groupby(phi["annee"] < COUPURE):
    etiquette = "avant" if avant else "apres"
    print("  %-6s %s : %+.3f  (n = %d)" % (etiquette, COUPURE,
          g["chomage"].corr(g["inflation"]), len(g)))

**Réponse 8.** Sur l'ensemble de la période, le nuage n'a **aucune forme claire**. La relation décroissante que la courbe de Phillips postule — plus de chômage, moins d'inflation — ne se lit pas.

**Réponse 9.** En séparant avant et après 2021, deux nuages distincts apparaissent. La période récente combine une inflation élevée et un chômage en baisse, ce qui la place ailleurs que les années 2000 et 2010.

**Réponse 10.** Comparez les trois coefficients imprimés ci-dessus. Si le signe change d'une sous-période à l'autre, ou si la corrélation d'ensemble est proche de zéro alors que celles des sous-périodes ne le sont pas, la conclusion est la même :

> **La relation n'est pas stable.** Un coefficient calculé sur l'ensemble mélange des régimes différents, et ne décrit aucun d'entre eux.

C'est exactement ce que la séance 6 annonçait : si le choix des bornes change la conclusion, c'est le choix qui fait la conclusion.

**Réponse 11.** Le troisième — celui qui distingue les périodes. Il montre à la fois la relation et son déplacement, là où le premier cache le second.

À côté, il faut écrire : **la période retenue, et pourquoi**. Ici, parce que la série des emplois vacants ne commence qu'en 2011 — une contrainte de données, non un choix d'analyse. Cela se dit.

## Pour ceux qui ont terminé

In [ ]:
lisse = bev.copy()
lisse["vacances"] = lisse["vacances"].rolling(4).mean()
lisse["chomage"]  = lisse["chomage"].rolling(4).mean()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(bev["chomage"], bev["vacances"], marker="o", markersize=3, lw=1)
ax[0].set_title("Séries brutes")
ax[1].plot(lisse["chomage"], lisse["vacances"], marker="o", markersize=3, lw=1, color="green")
ax[1].set_title("Moyenne mobile sur 4 trimestres")
for a in ax:
    a.set_xlabel("taux de chômage (%)"); a.set_ylabel("taux d'emplois vacants (%)")
plt.tight_layout(); plt.show()

**Réponse 12.** Le lissage retire les oscillations de court terme et rend le **déplacement** plus lisible. Mais il retarde aussi la détection : une moyenne sur quatre trimestres ne montre un changement qu'un an après qu'il a commencé.

C'est le compromis habituel : on gagne en lisibilité ce qu'on perd en réactivité. Sur une question de datation — « quand la courbe s'est-elle déplacée ? » —, c'est un défaut sérieux, et il faut le dire.

**Réponse 13.** Il suffit de remplacer `FR` par `DE` dans les trois adresses. L'intérêt de la comparaison : si les deux pays se déplacent au même moment, l'explication est probablement commune — la crise sanitaire. Si l'un se déplace seul, elle est nationale.

**Réponse 14.** Une série plus longue permettrait de voir si le déplacement de 2021 est exceptionnel, ou s'il s'en est produit d'autres — après 2008, par exemple. Avec quatorze ans de données, on ne peut pas trancher : **l'absence de précédent dans l'échantillon n'est pas l'absence de précédent.**

## Erreurs fréquentes

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Croiser mensuel et trimestriel | `merge` renvoie zéro ligne | Les dates doivent avoir le même pas |
| Sommer l'inflation au lieu de la moyenner | des taux à 20 ou 30 % | On somme les flux, on moyenne les taux |
| S'en tenir au nuage | le déplacement reste invisible | Une série temporelle garde son ordre |
| Lire une corrélation d'ensemble | « pas de relation » | Elle mélange des régimes différents |
| Oublier de dire la période | figure incomplète | La contrainte de données se déclare |